# Extract CSVs from ZIP on a Unity Catalog Volume → Spark DataFrames

The ZIP `1-10_AUG_Opik_Data.zip` sits on a Unity Catalog **Volume**, which is FUSE-mounted, so the driver can read/write it with ordinary Python file paths (`/Volumes/...`).

This notebook:
1. Lists what's inside the ZIP (confirm the 5 CSVs before extracting).
2. Extracts the CSVs into a subfolder on the Volume.
3. Reads each CSV into its own Spark DataFrame.

## Cell 1 — Config

In [ ]:
import os
import zipfile

# Full path to the ZIP on the Volume
ZIP_PATH     = "/Volumes/sindhu_db_prod/default/payment_raw/cc_cqr_opiklogs_2/1-10_AUG_Opik_Data.zip"

# Where to extract the CSVs (a new subfolder on the same Volume)
EXTRACT_DIR  = "/Volumes/sindhu_db_prod/default/payment_raw/cc_cqr_opiklogs_2/extracted"

os.makedirs(EXTRACT_DIR, exist_ok=True)
print("ZIP exists:", os.path.exists(ZIP_PATH))
print("Extract dir:", EXTRACT_DIR)

## Cell 2 — Peek inside the ZIP (no extraction yet)

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as z:
    members = [m for m in z.namelist() if not m.endswith("/")]
    for m in members:
        info = z.getinfo(m)
        print(f"{m}   ({info.file_size / 1e6:,.2f} MB uncompressed)")

csv_members = [m for m in members if m.lower().endswith(".csv")]
print(f"\nFound {len(csv_members)} CSV file(s).")

## Cell 3 — Extract only the CSV files to the Volume

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as z:
    for m in csv_members:
        # flatten any internal folder structure so all CSVs land directly in EXTRACT_DIR
        target_name = os.path.basename(m)
        target_path = os.path.join(EXTRACT_DIR, target_name)
        with z.open(m) as src, open(target_path, "wb") as dst:
            dst.write(src.read())
        print("extracted ->", target_path)

print("\nFiles now in extract dir:")
for f in sorted(os.listdir(EXTRACT_DIR)):
    print(" ", f)

## Cell 4 — Read each CSV into its own Spark DataFrame

DataFrames are stored in a dict `dfs` keyed by a clean table name (filename without `.csv`), and also bound as individual variables so you can use them directly.

In [ ]:
import re

def clean_name(fname: str) -> str:
    stem = os.path.splitext(os.path.basename(fname))[0]
    # make a valid identifier: lowercase, non-alphanumerics -> underscore
    name = re.sub(r"[^0-9a-zA-Z]+", "_", stem).strip("_").lower()
    if name and name[0].isdigit():
        name = "t_" + name
    return name

dfs = {}
csv_paths = [os.path.join(EXTRACT_DIR, f) for f in os.listdir(EXTRACT_DIR) if f.lower().endswith(".csv")]

for path in sorted(csv_paths):
    name = clean_name(path)
    df = (
        spark.read
             .option("header", True)
             .option("inferSchema", True)
             .option("multiLine", True)     # handles quoted newlines inside fields
             .option("escape", '"')
             .csv(path)
    )
    dfs[name] = df
    globals()[name] = df   # e.g. now usable as `df_payments`, etc.
    print(f"{name:40s}  rows={df.count():>10,}  cols={len(df.columns)}")

print("\nDataFrame variable names:", list(dfs.keys()))

## Cell 5 — Inspect one DataFrame

In [ ]:
# Show the first DataFrame (replace the key with the one you want)
first_name = list(dfs.keys())[0]
print("Schema of:", first_name)
dfs[first_name].printSchema()
display(dfs[first_name].limit(20))

## Cell 6 (optional) — Persist as managed tables

Uncomment to save each DataFrame as a table in the catalog so you can query it with SQL later.

In [ ]:
# TARGET_SCHEMA = "sindhu_db_prod.default"   # catalog.schema to write into
#
# for name, df in dfs.items():
#     full_name = f"{TARGET_SCHEMA}.{name}"
#     df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
#     print("saved table ->", full_name)